# Stage 9 - before/after visualization comparison

Use this notebook around a Stage 7/8 algorithm change. Run Stages 7 and 8 with the old algorithm, load the current result below, and store `previous`. After changing the algorithm and rerunning Stages 7 and 8, rerun the loading cell and store `new`. Finally, run the comparison section.

In [ ]:
import numpy as np
import pandas as pd
from importlib import import_module, reload

from src.api import prepare_visualization_data
from src.io import (
    PipelinePaths,
    load_npy_time_series,
    load_processed_dataset_inputs,
    load_stage8_outputs,
    open_sample,
)

comparison_helpers = reload(
    import_module("src.09_visualization.step05_comparison")
)
napari_layers = reload(
    import_module("src.09_visualization.napari_layers")
)
prepare_stage9_snapshot = comparison_helpers.prepare_stage9_snapshot
save_stage9_snapshot = comparison_helpers.save_stage9_snapshot
load_stage9_snapshot = comparison_helpers.load_stage9_snapshot
compare_stage9_snapshots = comparison_helpers.compare_stage9_snapshots
add_track_group = napari_layers.add_track_group

SAMPLE_ID = "44b6_0113de3b"
COMPARISON_NAME = "graph_tracking_disabled_vs_apply"
BOUNDARY_MARGIN_UM = 4.0

PREVIOUS_LABEL = "Graph disabled"
NEW_LABEL = "Graph apply"

OVERWRITE_PREVIOUS = False
OVERWRITE_NEW = False

paths = PipelinePaths.discover()
comparison_root = paths.stage9_comparisons / COMPARISON_NAME / SAMPLE_ID
previous_snapshot_dir = comparison_root / "previous"
new_snapshot_dir = comparison_root / "new"

## Load the current Stage 8 result

Rerun this cell after each Stage 7/8 run, before storing the corresponding snapshot.

In [ ]:
inputs = load_processed_dataset_inputs(SAMPLE_ID, paths=paths)
stage8 = load_stage8_outputs(paths=paths)
cells = pd.concat(
    [frame.assign(frame=index) for index, frame in enumerate(inputs.time_frames)],
    ignore_index=True,
)
raw = open_sample(paths.sample_zarr(SAMPLE_ID))
preprocessed, _ = load_npy_time_series(inputs.root / "preprocessing")
binary_mask, _ = load_npy_time_series(inputs.root / "masking")
instance_labels, _ = load_npy_time_series(inputs.root / "segmentation")

visualization = prepare_visualization_data(
    stage8.tracks,
    cells,
    spatial_shape_zyx=raw.shape[-3:],
    boundary_margin_um=BOUNDARY_MARGIN_UM,
)
endpoint_groups = visualization.endpoint_groups
if endpoint_groups is None:
    raise RuntimeError("Stage 9 endpoint groups were not prepared")

current_snapshot = prepare_stage9_snapshot(
    endpoint_groups,
    sample_id=SAMPLE_ID,
    boundary_margin_um=BOUNDARY_MARGIN_UM,
    voxel_size_zyx=visualization.voxel_size_zyx,
    spatial_shape_zyx=raw.shape[-3:],
    stage8_metadata=stage8.metadata,
)
SCALE_TZYX = (1.0, *visualization.voxel_size_zyx)
print("Current ended tracks:", current_snapshot.metadata["ended_track_count"])
print("Current new tracks:", current_snapshot.metadata["new_track_count"])

## 1. Store the current result as `previous`

Run this only after loading results produced by the old Stage 7/8 algorithm.

In [ ]:
save_stage9_snapshot(
    current_snapshot,
    previous_snapshot_dir,
    overwrite=OVERWRITE_PREVIOUS,
)
print(f"Stored {PREVIOUS_LABEL!r} snapshot at {previous_snapshot_dir}")

## 2. Store the current result as `new`

After changing the algorithm and rerunning Stages 7 and 8, rerun **Load the current Stage 8 result** above and then run this section.

In [ ]:
save_stage9_snapshot(
    current_snapshot,
    new_snapshot_dir,
    overwrite=OVERWRITE_NEW,
)
print(f"Stored {NEW_LABEL!r} snapshot at {new_snapshot_dir}")

## 3. Load both snapshots, compare them, and launch Napari

In [ ]:
previous_snapshot = load_stage9_snapshot(previous_snapshot_dir)
new_snapshot = load_stage9_snapshot(new_snapshot_dir)
comparison = compare_stage9_snapshots(previous_snapshot, new_snapshot)
SCALE_TZYX = (1.0, *previous_snapshot.metadata["voxel_size_zyx"])

print("Previous ended tracks:", comparison.summary["previous_ended_tracks"])
print("New ended tracks:", comparison.summary["new_ended_tracks"])
print("Removed ended tracks:", comparison.summary["removed_ended_tracks"])
print("Added ended tracks:", comparison.summary["added_ended_tracks"])
print("Previous new tracks:", comparison.summary["previous_new_tracks"])
print("New new tracks:", comparison.summary["new_new_tracks"])
print("Removed new tracks:", comparison.summary["removed_new_tracks"])
print("Added new tracks:", comparison.summary["added_new_tracks"])

In [ ]:
import napari

viewer = napari.Viewer(ndisplay=3)
raw_contrast_limits = [
    float(np.percentile(raw, 1)),
    float(np.percentile(raw, 99.8)),
]
viewer.add_image(
    raw,
    name="Raw Volume",
    scale=SCALE_TZYX,
    rendering="mip",
    colormap="gray",
    contrast_limits=raw_contrast_limits,
    visible=True,
)
viewer.add_image(
    preprocessed,
    name="Preprocessed Volume",
    scale=SCALE_TZYX,
    rendering="mip",
    colormap="gray",
    contrast_limits=(0.0, 1.0),
    visible=False,
)
viewer.add_labels(
    binary_mask,
    name="Binary Mask",
    scale=SCALE_TZYX,
    visible=False,
)
viewer.add_labels(
    instance_labels,
    name="Instance Labels",
    scale=SCALE_TZYX,
    visible=False,
)

comparison_groups = (
    (previous_snapshot.ended_tracks, "Previous | Ended Tracks", "Previous | Ended Centroids", "red", False),
    (previous_snapshot.new_tracks, "Previous | New Tracks", "Previous | New Centroids", "lime", False),
    (new_snapshot.ended_tracks, "New | Ended Tracks", "New | Ended Centroids", "red", False),
    (new_snapshot.new_tracks, "New | New Tracks", "New | New Centroids", "lime", False),
    (comparison.removed_ended_tracks, "Removed | Ended Tracks", "Removed | Ended Centroids", "red", True),
    (comparison.removed_new_tracks, "Removed | New Tracks", "Removed | New Centroids", "lime", True),
    (comparison.added_ended_tracks, "Added | Ended Tracks", "Added | Ended Centroids", "red", True),
    (comparison.added_new_tracks, "Added | New Tracks", "Added | New Centroids", "lime", True),
)
for frame, track_name, point_name, color, visible in comparison_groups:
    add_track_group(
        viewer,
        frame,
        track_name=track_name,
        point_name=point_name,
        color=color,
        scale=SCALE_TZYX,
        visible=visible,
        tail_length=20,
    )

napari.run()